# LiteraLevel · discursos de la Casa Blanca de Obama → Drive

Vídeos oficiales de obamawhitehouse.archives.gov, marcados como **dominio
público** (obra del gobierno federal de EE. UU.). Se bajan directos a
`MyDrive/literalevel_cara/casa_blanca/` y se miden con MediaPipe (blendshapes
ARKit, pose de la cabeza, tono y energía). Reanudable: si una sesión se cae,
se vuelve a ejecutar y sigue donde iba.

Antes: subir `lista.csv` (de `listar_casa_blanca.py`) con «Upload to Colab».

In [ ]:
# --- 1. Drive (pide permiso la primera vez) y dependencias
from google.colab import drive
drive.mount('/content/drive')
!pip -q install mediapipe==0.10.21 librosa imageio-ffmpeg
!wget -q -nc -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
import os, csv
DESTINO = '/content/drive/MyDrive/literalevel_cara/casa_blanca'
os.makedirs(f'{DESTINO}/videos', exist_ok=True); os.makedirs(f'{DESTINO}/medidas', exist_ok=True)
lista = [f for f in ['/content/lista.csv', f'{DESTINO}/lista.csv'] if os.path.exists(f)][0]
!cp -n {lista} {DESTINO}/lista.csv
filas = list(csv.DictReader(open(lista, encoding='utf8')))
TIPOS = {'discurso_semanal', 'presidente'}
sel = [f for f in filas if f['tipo'] in TIPOS and f['mp4'] and f['dominio_publico'] == '1']
print(len(filas), 'vídeos en la lista;', len(sel), 'seleccionados (discursos del presidente, dominio público)')

In [ ]:
# --- 2. Bajar a Drive (varios a la vez; se salta lo ya bajado)
import concurrent.futures, subprocess
def bajar(f):
    nombre = f['fecha'] + '_' + f['slug'].rstrip('/').split('/')[-1][:80] + '.mp4'
    ruta = f'{DESTINO}/videos/{nombre}'
    if os.path.exists(ruta) and os.path.getsize(ruta) > 1e5:
        return 'ya'
    r = subprocess.run(['wget', '-q', '-O', ruta + '.part', f['mp4']])
    if r.returncode == 0:
        os.replace(ruta + '.part', ruta); return 'ok'
    if os.path.exists(ruta + '.part'): os.remove(ruta + '.part')
    return 'fallo'
from collections import Counter
with concurrent.futures.ThreadPoolExecutor(6) as ex:
    res = Counter(ex.map(bajar, sel))
print(res)
!du -sh {DESTINO}/videos

In [ ]:
MEDIR_PY = "\"\"\"Expresión y prosodia de cada fotograma de un clip de cara hablando.\n\nPor cada vídeo escribe un .npz con\n  t        segundos de cada fotograma\n  bs       las 52 blendshapes ARKit de MediaPipe (nombres en `bs_nombres`)\n  pose     giro de la cabeza en grados: guiñada, cabeceo, alabeo\n  ok       1 si se encontró cara\n  f0       tono en Hz (0 si no hay voz), llevado a los fotogramas\n  db       energía RMS en dB, llevada a los fotogramas\n\nEs la parte de `modelo_articulatorio/medir_video.py` que sirve para la vida\nde la cara (parpadeo, cejas, cabeza), más la pose y la prosodia. Si hay\nvarias caras se queda con la más grande. Reanudable.\n\nUso:\n  python medir_expresion.py --modelo face_landmarker.task --salida DIR clip1.mp4 [...]\n\"\"\"\nimport argparse\nimport os\nimport subprocess\nimport tempfile\n\nimport cv2\nimport imageio_ffmpeg\nimport librosa\nimport mediapipe as mp\nimport numpy as np\nfrom mediapipe.tasks import python as mp_python\nfrom mediapipe.tasks.python import vision\n\n\ndef pose_de_matriz(m):\n    \"\"\"Ángulos (guiñada, cabeceo, alabeo) en grados de la matriz 4x4 de MediaPipe.\"\"\"\n    r = np.asarray(m)[:3, :3]\n    cabeceo = np.degrees(np.arctan2(r[2, 1], r[2, 2]))\n    guinada = np.degrees(np.arcsin(-np.clip(r[2, 0], -1, 1)))\n    alabeo = np.degrees(np.arctan2(r[1, 0], r[0, 0]))\n    return guinada, cabeceo, alabeo\n\n\ndef tiene_video(ruta):\n    \"\"\"Algunos tramos bajan solo con audio; cv2 se cuelga con ellos (visto el 25 sep).\"\"\"\n    r = subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), '-hide_banner', '-i', ruta],\n                       capture_output=True, text=True, errors='replace')\n    return 'Video:' in r.stderr\n\n\ndef prosodia(video, t):\n    with tempfile.TemporaryDirectory() as d:\n        wav = os.path.join(d, 'a.wav')\n        subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), '-loglevel', 'error', '-y', '-i', video,\n                        '-ac', '1', '-ar', '16000', wav], check=True)\n        y, sr = librosa.load(wav, sr=16000)\n    salto = 160  # 10 ms\n    f0, voz, _ = librosa.pyin(y, fmin=65, fmax=400, sr=sr, frame_length=1024, hop_length=salto)\n    f0 = np.where(voz, f0, 0.0)\n    rms = librosa.feature.rms(y=y, frame_length=400, hop_length=salto)[0]\n    db = 20 * np.log10(rms + 1e-5)\n    ta = np.arange(len(f0)) * salto / sr\n    tr = np.arange(len(db)) * salto / sr\n    # f0 al fotograma: el valor más cercano (no interpolar entre voz y silencio)\n    idx = np.clip(np.searchsorted(ta, t), 0, len(f0) - 1)\n    return f0[idx], np.interp(t, tr, db)\n\n\ndef medir(video, lmk, salida):\n    cap = cv2.VideoCapture(video)\n    fps = cap.get(cv2.CAP_PROP_FPS) or 25\n    t, bs, pose, ok = [], [], [], []\n    nombres = None\n    i = 0\n    while True:\n        bien, img = cap.read()\n        if not bien:\n            break\n        ts = int(i * 1000 / fps)\n        r = lmk.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,\n                                          data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), ts)\n        t.append(i / fps)\n        if r.face_blendshapes:\n            # la cara más grande: la de mayor anchura de la malla\n            k = int(np.argmax([np.ptp([p.x for p in c]) for c in r.face_landmarks]))\n            cats = r.face_blendshapes[k]\n            nombres = nombres or [c.category_name for c in cats]\n            bs.append([c.score for c in cats])\n            pose.append(pose_de_matriz(r.facial_transformation_matrixes[k]))\n            ok.append(1)\n        else:\n            bs.append([0.0] * 52)\n            pose.append((0.0, 0.0, 0.0))\n            ok.append(0)\n        i += 1\n    cap.release()\n    t = np.array(t)\n    f0, db = prosodia(video, t)\n    np.savez_compressed(salida, t=t, bs=np.array(bs, np.float32), pose=np.array(pose, np.float32),\n                        ok=np.array(ok, np.uint8), f0=f0.astype(np.float32), db=db.astype(np.float32),\n                        bs_nombres=np.array(nombres or []), fps=fps)\n    return len(t), float(np.mean(ok)) if ok else 0.0\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--modelo', required=True)\n    ap.add_argument('--salida', required=True)\n    ap.add_argument('videos', nargs='+')\n    a = ap.parse_args()\n    os.makedirs(a.salida, exist_ok=True)\n    for v in a.videos:\n        destino = os.path.join(a.salida, os.path.splitext(os.path.basename(v))[0] + '.npz')\n        if os.path.exists(destino):\n            continue\n        if not tiene_video(v):\n            print(f'{os.path.basename(v)}: sin imagen, se salta')\n            continue\n        opts = vision.FaceLandmarkerOptions(\n            base_options=mp_python.BaseOptions(model_asset_path=a.modelo),\n            running_mode=vision.RunningMode.VIDEO, num_faces=2,\n            output_face_blendshapes=True, output_facial_transformation_matrixes=True)\n        with vision.FaceLandmarker.create_from_options(opts) as lmk:\n            n, frac = medir(v, lmk, destino)\n        print(f'{os.path.basename(v)}: {n} fotogramas, cara en {frac:.0%}')\n\n\nif __name__ == '__main__':\n    main()\n"
# --- 3. Medir (MediaPipe en CPU; varios procesos a la vez)
open('/content/medir_expresion.py', 'w', encoding='utf8').write(MEDIR_PY)
import glob
videos = sorted(glob.glob(f'{DESTINO}/videos/*.mp4'))
faltan = [v for v in videos if not os.path.exists(f"{DESTINO}/medidas/{os.path.basename(v)[:-4]}.npz")]
print(len(videos), 'vídeos,', len(faltan), 'por medir')
trozos = [faltan[i::4] for i in range(4)]
procs = [subprocess.Popen(['python', '/content/medir_expresion.py', '--modelo', '/content/face_landmarker.task',
                          '--salida', f'{DESTINO}/medidas', *t], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
         for t in trozos if t]
for p in procs:
    for linea in p.stdout:
        if 'fotogramas' in linea or 'salta' in linea: print(linea.strip())
!ls {DESTINO}/medidas | wc -l